In [1]:
# Import necessary libraries and define metadata for the book to be processed

import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
import json

# Metadata for the book to be processed

book_metadata =  {
        "title": "The Mind and Its Education", 
        "author" : "George Herbert Betts",
        "source_url": "https://www.gutenberg.org/ebooks/20220",
        "filename": "themind.txt"
  }

In [2]:
# Initialize OpenAI client and define a function to get completions from the model

client_openai = OpenAI()

def get_completion(prompt):
    response = client_openai.chat.completions.create(
        model="gpt-6-luna",
        messages=[
            {"role":"system", "content":"You are a helpful assistant connected to a database for document search."},
            {"role":"user", "content": prompt},
        ]
    )
    return response.choices[0].message.content


In [3]:
# Split the text of the book into manageable chunks for processing

text_splitter = RecursiveCharacterTextSplitter(
    separators=[". ", "? ", "! "],
    chunk_size=2000,
    chunk_overlap=300,
)

with open("./data/themind.txt", "r", encoding="utf-8") as file:
    content = file.read()
chunks = text_splitter.create_documents([content])


In [4]:
# Initialize Chroma client and create or get the collection for storing document embeddings

client_chroma = chromadb.PersistentClient("./advanced")
collection = client_chroma.get_or_create_collection(name="advanced", metadata={"hnsw:space": "cosine"})

In [5]:
# Populate the RAG collection with chunks of the book and their metadata

for idx, chunk in enumerate(chunks):
    doc_text = chunk.page_content
    book_metadata["chunk_idx"] = str(idx)
    collection.add(
        documents=[doc_text],
        ids=[f"{book_metadata['title']}_{idx}"],
        metadatas=[book_metadata]
    )

In [6]:
# Populate the RAG query with search results and their metadata

def populate_rag_query(query, n_results=1):
    search_results = collection.query(query_texts=[query], n_results=n_results)
    result_str = ""
    documents = search_results.get("documents")
    metadatas = search_results.get("metadatas")

    if not documents or not documents[0]:
        return result_str

    document_list = documents[0]
    metadata_list = metadatas[0] if metadatas and metadatas[0] else []

    for idx, result in enumerate(document_list):
        if result is None:
            continue

        metadata = metadata_list[idx] if idx < len(metadata_list) else None
        metadata = metadata or {}

        formatted_result = f"""<SEARCH RESULT>
        <DOCUMENT>{result}</DOCUMENT>
        <METADATA>
        <TITLE>{metadata.get("title", "")}</TITLE>
        <AUTHOR>{metadata.get("author", "")}</AUTHOR>
        <CHUNK_IDX>{metadata.get("chunk_idx", "")}</CHUNK_IDX>
        <URL>{metadata.get("source_url", "")}</URL>
        </METADATA>
        </SEARCH RESULT>"""
        result_str += formatted_result
    return result_str

In [7]:
# Generate a RAG prompt using the user query and search results

def make_rag_prompt(query, results):
    return f"""<INSTRUCTIONS>
   <EXAMPLE CITATION>
   Answer to the user query in your own words, drawn from the search results.
   - "Direct quote from source material backing up the claim" - [Source: Title, Author, Chunk: chunk index, Link: url]
   </EXAMPLE CITATION>
   </INSTRUCTIONS>

    <USER QUERY>
    {query}
    </USER QUERY>

    <SEARCH RESULTS>
    {results}
    </SEARCH RESULTS>
    
    Your answer:"""

# **Decoupled RAG prompt**

In [8]:
# To retrieve an entry by id, pass the id to get()
collection.get('The Mind and Its Education_151')

{'ids': ['The Mind and Its Education_151'],
 'embeddings': None,
 'documents': ['. This statement is, of course, but a particular application of the law of attention. Interest brings the whole self into action. Under its urging the mind is active and alert. The new facts learned are completely157 registered, and are assimilated to other facts to which they are related. Many associative connections are formed, hence the new matter is more certain of recall, and possesses more significance and meaning.\n\n**Association and Methods of Learning.**—The number and quality of our associations depends in no small degree on our methods of learning. We may be satisfied merely to impress what we learn on our memory, committing it uncritically as so many facts to be stored away as a part of our education. We may go a step beyond this and grasp the simplest and most obvious meanings, but not seek for the deeper and more fundamental relations. We may learn separate sections or divisions of a subject

In [9]:
# To retrieve an entry by a field value, pass the field name and value to get() like so
collection.get(where={"chunk_idx": {"$eq": "151"}})

{'ids': ['The Mind and Its Education_151'],
 'embeddings': None,
 'documents': ['. This statement is, of course, but a particular application of the law of attention. Interest brings the whole self into action. Under its urging the mind is active and alert. The new facts learned are completely157 registered, and are assimilated to other facts to which they are related. Many associative connections are formed, hence the new matter is more certain of recall, and possesses more significance and meaning.\n\n**Association and Methods of Learning.**—The number and quality of our associations depends in no small degree on our methods of learning. We may be satisfied merely to impress what we learn on our memory, committing it uncritically as so many facts to be stored away as a part of our education. We may go a step beyond this and grasp the simplest and most obvious meanings, but not seek for the deeper and more fundamental relations. We may learn separate sections or divisions of a subject

In [10]:
# One thing to keep in mind in our implementation, when an id is not found in the collection, a dictionary is still returned.
collection.get(where={"chunk_idx": {"$eq": "999999999"}})

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [ ]:
# Retrieve the previous and next chunks based on the current chunk index
def get_previous_and_next_chunks(chunk_idx):
    chunk_idx = int(chunk_idx)
    previous_chunk = collection.get(where={"chunk_idx": { "$eq": str(chunk_idx - 1)}})
    next_chunk = collection.get(where={"chunk_idx": { "$eq": str(chunk_idx + 1)}})
    
    return previous_chunk, next_chunk

In [12]:
# Retrieve and format the previous, current, and next chunks for a given original chunk

def expanded_search_results(original_chunk):
    original_chunk_idx = int(original_chunk["metadatas"][0]["chunk_idx"])
    previous_chunk, next_chunk = get_previous_and_next_chunks(original_chunk_idx)
    result_str = ""
    for chunk in [previous_chunk, original_chunk, next_chunk]:
        if len(chunk["metadatas"])>0:
            metadata = chunk["metadatas"][0]
            formatted_result = f"""<SEARCH RESULT>
            <DOCUMENT>{chunk["documents"][0]}</DOCUMENT>
            <METADATA>
            <TITLE>{metadata.get("title", "")}</TITLE>
            <AUTHOR>{metadata.get("author", "")}</AUTHOR>
            <CHUNK_IDX>{metadata.get("chunk_idx", "")}</CHUNK_IDX>
            <URL>{metadata.get("source_url", "")}</URL>
            </METADATA>
            </SEARCH RESULT>"""
            result_str += formatted_result
    return result_str

original_demo_chunk = collection.get(where={"chunk_idx": {"$eq": "151"}})
expanded_results = expanded_search_results(original_demo_chunk)
print(expanded_results)

<SEARCH RESULT>
            <DOCUMENT>. The little child of two years may not understand the gravity of the offense in tearing the leaves out of books, but if its hands are sharply spatted whenever they tear a book, the association between the sight of books and tearing them will soon cease. In fact, all punishment should have for its156 object the use of pain in the breaking of associative bonds between certain situations and wrong responses to them.

On the other hand, the dog that is being trained to perform his tricks is rewarded with a tidbit or a pat when the right response has been made. In this way the bond for this particular act is strengthened through the use of pleasure. All matter studied and learned under the stimulus of good feeling, enthusiasm, or a pleasurable sense of victory and achievement not only tends to set up more permanent and valuable associations than if learned under opposite conditions, but it also exerts a stronger appeal to our interest and appreciation.

In [13]:
# Function to create a decoupled RAG prompt using search results and a query

def make_decoupled_rag_prompt(query, n_results=1):
    search_results = collection.query(query_texts=[query], n_results=n_results)
    total_result_str = ""
    documents = search_results.get("documents")
    metadatas = search_results.get("metadatas")

    if documents and documents[0] and metadatas and metadatas[0]:
        for doc, metadata in zip(documents[0], metadatas[0]):
            if doc is None or metadata is None:
                continue

            chunk = {
                "documents": [doc],
                "metadatas": [metadata],
            }
            total_result_str += expanded_search_results(chunk)

    rag_prompt = make_rag_prompt(query, total_result_str)
    return rag_prompt

# given a search result, we can get everything back in a formatted string
prompt = make_decoupled_rag_prompt("What is the best way to learn a complex, challenging subject?")
rag_completion = get_completion(prompt)
print(rag_completion)

A good way to learn a difficult subject is to **combine broad exposure with active thinking and practice**:

1. **Build a strong foundation.** Learn from varied sources and examples, not just one kind of material. Betts emphasizes deliberately expanding your experience and keeping what you learn fresh through recall and reconstruction. [Source: *The Mind and Its Education*, George Herbert Betts, Chunk: 135, Link: https://www.gutenberg.org/ebooks/20220]
2. **Don’t only collect information.** Pause to connect ideas and work out what they mean. As Betts puts it, learners can be “so taken up with receiving that they never perform the higher act of combining.” [Source: *The Mind and Its Education*, George Herbert Betts, Chunk: 136, Link: https://www.gutenberg.org/ebooks/20220]
3. **Make something with what you learn.** Explain a concept in your own words, solve problems, or apply it to a project. Betts recommends building from your own material and making ideas real through activity. [Sourc

# **Contextual Query Retrieval (CQR)**

In [14]:
# Example chat memory and contextual query for testing Contextual Query Retrieval (CQR) in RAG prompts

chat_memory = [
    {"role": "user", "content": "Tell me about the nature of consciousness."},
    {"role": "assistant", "content": "Consciousness is like a stream that flows from birth to death. It's a continuous process that we can only observe in the present moment. The content of consciousness includes our thoughts, feelings, and will - the processes of knowing, feeling, and choosing."},
    {"role": "user", "content": "That's interesting. How does attention relate to consciousness?"},
    {"role": "assistant", "content": "Attention is the concentration of consciousness on a particular object or thought. It's like a wave in the stream of consciousness, where the object of attention stands out more prominently. Attention allows us to focus our mental energy, making the object of focus clearer and more defined in our mind."},
    {"role": "user", "content": "I've heard attention is important for mental efficiency. Is that true?"},
    {"role": "assistant", "content": "Yes, attention is crucial for mental efficiency. It's like focusing sunlight through a lens - it concentrates mental energy, allowing us to accomplish more in less time. Strong attention skills can significantly enhance our ability to learn, solve problems, and be productive in our mental tasks."}
]

contextual_query = "How does it affect our ability to think and work effectively?"

In [15]:
# Convert chat memory into a formatted chat history string

chat_history = ""

for message in chat_memory:
    role = message["role"]
    content = message["content"]
    chat_history += f"{role}: {content}\n\n"

print("Chat History:")
print(chat_history)

Chat History:
user: Tell me about the nature of consciousness.

assistant: Consciousness is like a stream that flows from birth to death. It's a continuous process that we can only observe in the present moment. The content of consciousness includes our thoughts, feelings, and will - the processes of knowing, feeling, and choosing.

user: That's interesting. How does attention relate to consciousness?

assistant: Attention is the concentration of consciousness on a particular object or thought. It's like a wave in the stream of consciousness, where the object of attention stands out more prominently. Attention allows us to focus our mental energy, making the object of focus clearer and more defined in our mind.

user: I've heard attention is important for mental efficiency. Is that true?

assistant: Yes, attention is crucial for mental efficiency. It's like focusing sunlight through a lens - it concentrates mental energy, allowing us to accomplish more in less time. Strong attention sk

In [16]:
# Function to rewrite a query in the context of chat history for Contextual Query Retrieval (CQR)

def rewrite_query(query, chat_history):
    prompt = f"""<INSTRUCTIONS>
    Transform the user's latest message into an optimal retrieval query.

    Guidelines:
    - Resolve all references, pronouns, and omitted subjects.
    - Incorporate only context necessary to understand the request.
    - Preserve all technical terminology.
    - Expand abbreviated references when the meaning is clear from context.
    - Include keywords and concepts likely to appear in relevant documents.
    - Remove greetings, filler language, and conversational artifacts.
    - Do not answer the question.
    - Do not summarize the conversation.
    - Do not generate explanatory text.

    Output a single standalone search query.
    </INSTRUCTIONS>

    <CHAT_HISTORY>
    {chat_history}
    </CHAT_HISTORY>

    <LATEST_QUERY>
    {query}
    </LATEST_QUERY>

    Your rewritten query:"""
    
    return get_completion(prompt)

cqr_query = rewrite_query(contextual_query, chat_history)
print("ORIGINAL QUERY:")
print(contextual_query)
print("CQR QUERY:")
print(cqr_query)

ORIGINAL QUERY:
How does it affect our ability to think and work effectively?
CQR QUERY:
How does attention affect mental efficiency and the ability to think, learn, solve problems, and work productively?


In [17]:
# Function to perform Contextual Query Retrieval (CQR) and Retrieval-Augmented Generation (RAG)

def perform_cqr_rag(query, chat_history, n_results=2):
    # Rewrite the query using the chat history
    refined_query = rewrite_query(query, chat_history)
    result_str = populate_rag_query(refined_query, n_results)
    rag_prompt = make_rag_prompt(refined_query, result_str)
    rag_completion = get_completion(rag_prompt)
    return refined_query, rag_completion

refined_query, rag_completion = perform_cqr_rag(contextual_query, chat_history)
print(rag_completion)

Attention directs mental energy by selecting which parts of experience receive emphasis. It makes the chosen thought or problem clearer and more prominent, helping you think about it rather than letting it blend into distractions. [Source: *The Mind and Its Education*, George Herbert Betts, Chunk: 14, Link: https://www.gutenberg.org/ebooks/20220]

That focus can improve mental efficiency and productivity: Betts compares concentrated thought to sunlight focused through a lens, saying, “More can now be accomplished in minutes than before in hours.” [Source: *The Mind and Its Education*, George Herbert Betts, Chunk: 14, Link: https://www.gutenberg.org/ebooks/20220] Applied to learning and problem-solving, the idea is that sustained attention helps you work through material or a problem more effectively than scattered thought.

Attention also supports learning and memory. Betts says that material that interests us—and so attracts attention with less effort—is, other things being equal, “th

# **Hypothetical Document Embeddings (HYDE)**

In [18]:
# Function to create a hypothetical answer prompt for the HyDE (Hypothetical Document Embeddings) approach

def make_hyde_prompt(query):
    return f"""<INSTRUCTIONS>
    Generate a hypothetical passage from a knowledge base, article, technical document, or authoritative source that would answer the query.

    The passage should:
    - Contain relevant terminology and topic-specific vocabulary.
    - Explain concepts, mechanisms, causes, effects, examples, and related entities when appropriate.
    - Read like a real document that could exist in a search index.
    - Maximize semantic overlap with documents likely to contain the answer.
    - Avoid conversational language.
    - Avoid phrases such as "I think", "I don't know", or "as an AI".
    - Avoid making up precise facts unless they are implied by widely known domain knowledge.

    Write 1-3 paragraphs.
    </INSTRUCTIONS>

    <QUERY>{query}</QUERY>
    """

In [19]:
# Example of using the HyDE approach to generate a query and retrieve results from the RAG model

query = "How do we know what the mind is?"

hyde_query_prompt = make_hyde_prompt(query)
hyde_query = get_completion(hyde_query_prompt)

print(hyde_query)

if hyde_query is None:
    raise ValueError("HyDE query generation returned no content.")

results = collection.query(
    query_texts=[hyde_query],
    n_results=2
)
print(results)

The mind is known through a combination of first-person experience and third-person evidence. Introspection gives a person direct, though sometimes incomplete or mistaken, access to experiences such as thoughts, emotions, intentions, and sensations. Other people’s mental states cannot be observed directly; they are inferred from speech, behavior, and patterns of response, and studied alongside brain activity and changes caused by injury, development, or intervention.

In psychology and neuroscience, “mind” commonly refers to processes such as perception, memory, attention, emotion, reasoning, and consciousness. Researchers investigate these processes by comparing reports and behavior with measurable activity in the nervous system. These methods support explanations of how mental functions work, but they do not settle every philosophical question—for example, how subjective experience relates to physical brain processes. Thus, what is known about the mind depends on converging evidence,

In [20]:
# Function to answer a query using the HyDE approach and RAG model

def answer_query_with_hyde(user_query):
    hyde_prompt = make_hyde_prompt(user_query)
    hyde_query = get_completion(hyde_prompt)
    if hyde_query is None:
        raise ValueError("HyDE query generation returned no content.")
    result_str = populate_rag_query(hyde_query, n_results=3)
    rag_prompt = make_rag_prompt(user_query, result_str)
    rag_completion = get_completion(rag_prompt)
    
    return rag_completion

user_query = "How do we know what the mind is?"
answer = answer_query_with_hyde(user_query)
print(answer)

We know the mind most directly through **introspection**: noticing our own thoughts, feelings, memories, and decisions as they occur. Betts argues that descriptions in books or lectures are only useful when checked against our own experience: “The only way to know what mind is, is to look in upon our own consciousness and observe what is transpiring there.” [Source: *The Mind and Its Education*, George Herbert Betts, Chunk: 1, Link: https://www.gutenberg.org/ebooks/20220]

But this first-hand knowledge applies only to our own minds. We learn about other people’s minds indirectly, by interpreting their words, expressions, and actions: “I can never know your mind at all except through your bodily acts and expressions.” [Source: *The Mind and Its Education*, George Herbert Betts, Chunk: 0, Link: https://www.gutenberg.org/ebooks/20220]


# **FUSION SEARCH**

In [21]:
# Example of how to prompt the model for JSON, get it back, and parse it so we can use it as a regular Python dict

response = get_completion("""<INSTRUCTIONS>
Generate valid JSON with the key "data" and a list of 1,2,3.
Only generate the JSON. Do not output any additional characters.
Do not output markdown backticks. Just raw JSON.
</INSTRUCTIONS>
                          
YOUR JSON:""")
our_loaded_json = json.loads(str(response))
for number in our_loaded_json['data']:
    print(f"Number: {number}")

Number: 1
Number: 2
Number: 3


In [22]:
# Function to generate subquestions from a user query using a language model and parse the response as JSON

def generate_subquestions(query):
    prompt = f"""<INSTRUCTIONS>
    Given the following user query, generate a list of 2-4 subquestions that would help in answering the original query. 
    Return the result as JSON with the key "data" and the list as its value. 
    Only output valid JSON and no other characters.
    Do not output markdown backticks. Just output raw JSON only.
    </INSTRUCTIONS>

    <QUERY>{query}</QUERY>
    """
    response = get_completion(prompt)
    # if the model generates invalid json, uncomment the print line and inspect what went wrong
    # print(response)
    subquery_dict = json.loads(str(response))
    return subquery_dict['data']

generate_subquestions("What should I eat for dinner tonight?")

['Do you have any dietary restrictions, allergies, or food preferences?',
 'What ingredients do you already have available?',
 'How much time do you want to spend preparing dinner?',
 'Are you looking for something light, hearty, or a particular type of cuisine?']

In [23]:
# Function to get subquestions for a query, populate RAG context for each, and concatenate the Q&A pairs

def get_and_concat_subquestions(query):
    subquestions = generate_subquestions(query)
    subquestion_context = []
    for subquestion in subquestions:
        result_str = populate_rag_query(subquestion)
        rag_prompt = make_rag_prompt(subquestion, result_str)
        answer = get_completion(rag_prompt)
        subquestion_context.append(f"Q: {subquestion}\nA: {answer}")
    return "\n\n".join(subquestion_context)

In [24]:
# Function to perform a fusion search by generating subquestions, populating RAG context, and producing a final answer

def fusion_search(query):
    print(f"Original query: {query}")
    subquestion_context = get_and_concat_subquestions(query)
    print("Subquestion context:", subquestion_context)
    final_prompt = f"""<INSTRUCTIONS>
    Using the following information from subquestions, answer the original query.
    </INSTRUCTIONS>

    <SUBQUESTION_INFO>
    {subquestion_context}
    </SUBQUESTION_INFO>

    <ORIGINAL_QUERY>{query}</ORIGINAL_QUERY>

    Final Answer:"""

    final_answer = get_completion(final_prompt)
    return final_answer

user_query = "How should we approach learning, given what we know about the mind?"
result = fusion_search(user_query)

print("\nFinal Result:")
print(result)

Original query: How should we approach learning, given what we know about the mind?
Subquestion context: Q: What do we know about how attention, memory, and prior knowledge shape learning?
A: The source directly explains memory through **association**: “Without the machinery and processes of association we could have no memory.” It describes recall as relying on connections among different brain centers, and suggests that more complex learning requires more such connections. It also argues that children may not yet have the neural development needed for some complex associations. [Source: *The Mind and Its Education*, George Herbert Betts, Chunk: 140, Link: https://www.gutenberg.org/ebooks/20220]

This supports the idea that what a learner can connect to and recall matters for learning, and that readiness can affect how readily complex material is learned. The provided passage does **not** directly discuss attention or prior knowledge, so it can’t establish their specific effects.

Q: 